# Tiền xử lý dữ liệu

## Đọc dữ liệu

In [928]:
import pandas as pd
import numpy as np
df = pd.read_csv('data_raw.csv')
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 6007 entries, 0 to 6006
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   ma_tin          6007 non-null   int64
 1   ngay_dang       6007 non-null   str  
 2   duong           5750 non-null   str  
 3   phuong_xa       6007 non-null   str  
 4   quan_huyen      6007 non-null   str  
 5   gia             6007 non-null   str  
 6   dien_tich       6007 non-null   str  
 7   so_pn           5348 non-null   str  
 8   so_wc           5085 non-null   str  
 9   so_tang         5739 non-null   str  
 10  huong_nha       1435 non-null   str  
 11  huong_ban_cong  887 non-null    str  
 12  mat_tien        4475 non-null   str  
 13  duong_vao       4237 non-null   str  
 14  phap_ly         5972 non-null   str  
 15  noi_that        3760 non-null   str  
 16  URL             6007 non-null   str  
dtypes: int64(1), str(16)
memory usage: 797.9 KB


## Xóa các cột không dùng hoặc thiếu quá nhiều

In [929]:
cols_to_remove = ['ma_tin', 'ngay_dang', 'URL', 'duong']
df.drop(columns=cols_to_remove, inplace=True, errors='ignore')
missing_percentage = df.isnull().mean() * 100
print("Tỉ lệ missing value của các cột (%):")
print(missing_percentage.round(2).sort_values(ascending=False))

threshold = 50  # Ngưỡng 50%
cols_high_missing = missing_percentage[missing_percentage > threshold].index
print(f"\nĐang xóa các cột thiếu trên {threshold}% dữ liệu: {list(cols_high_missing)}")
df.drop(columns=cols_high_missing, inplace=True)

# Kiểm tra lại các cột còn lại
print("\nCác cột còn lại sau khi xử lý:")
print(df.columns.tolist())

Tỉ lệ missing value của các cột (%):
huong_ban_cong    85.23
huong_nha         76.11
noi_that          37.41
duong_vao         29.47
mat_tien          25.50
so_wc             15.35
so_pn             10.97
so_tang            4.46
phap_ly            0.58
dien_tich          0.00
gia                0.00
quan_huyen         0.00
phuong_xa          0.00
dtype: float64

Đang xóa các cột thiếu trên 50% dữ liệu: ['huong_nha', 'huong_ban_cong']

Các cột còn lại sau khi xử lý:
['phuong_xa', 'quan_huyen', 'gia', 'dien_tich', 'so_pn', 'so_wc', 'so_tang', 'mat_tien', 'duong_vao', 'phap_ly', 'noi_that']


## Xử lý các cột dạng số

In [930]:
def clean_gia(price_str):
    """Chuyển đổi chuỗi giá sang số thực (VNĐ)."""
    if pd.isna(price_str): return np.nan
    price_str = str(price_str).replace(' ', '').replace('.', '').replace(',', '.')
    try:
        if 'tỷ' in price_str:
            return float(price_str.replace('tỷ', '')) * 1_000_000_000
        elif 'triệu' in price_str:
            return float(price_str.replace('triệu', '')) * 1_000_000
    except ValueError:
        return np.nan
    return np.nan

def clean_m(area_str):
    """Làm sạch cột diện tích/mặt tiền/đường vào."""
    if pd.isna(area_str): return np.nan
    area_str = str(area_str).replace(' ', '').replace(',', '.').replace('m²', '').replace('m', '')
    try:
        return float(area_str)
    except ValueError:
        return np.nan

def clean_so_phong(text):
    """Trích xuất số tầng, phòng ngủ, WC."""
    if pd.isna(text): return np.nan
    try:
        clean_text = str(text).lower().replace('tầng', '').replace('phòng', '').strip()
        return float(clean_text)
    except ValueError:
        return np.nan

In [931]:
df['gia'] = df['gia'].apply(clean_gia)
df['dien_tich'] = df['dien_tich'].apply(clean_m)
df['mat_tien'] = df['mat_tien'].apply(clean_m)
df['duong_vao'] = df['duong_vao'].apply(clean_m)

# Xử lý các cột đếm
df['so_pn'] = df['so_pn'].apply(clean_so_phong)
df['so_wc'] = df['so_wc'].apply(clean_so_phong)
df['so_tang'] = df['so_tang'].apply(clean_so_phong)
print('Đã xử lý các cột dạng số')

Đã xử lý các cột dạng số


## Xử lý và đồng bộ dữ liệu cột pháp lý và cột nội thất

In [932]:
df['phap_ly'].value_counts()

phap_ly
Sổ đỏ/ Sổ hồng                  4152
Sổ đỏ/ Sổ hồng.                  819
Sổ hồng                          305
Sổ đỏ.                           139
Có sổ.                           138
                                ... 
Sổ hồng riêng, chính chủ bán       1
Sổ riêng                           1
Sổ đỏ / sổ hồng.                   1
Sổ hồng vuông vức.                 1
Sổ hồng hoàn công.                 1
Name: count, Length: 84, dtype: int64

In [933]:
df['noi_that'].value_counts()

noi_that
Đầy đủ                                                          1921
Cơ bản                                                           765
Đầy đủ.                                                          365
Cơ bản.                                                          155
Không nội thất                                                   154
                                                                ... 
Nhà thô                                                            1
Cơ bản nội thất đẹp xịn.                                           1
Nội thất xịn.                                                      1
Nội thất đẹp, hiện đại.                                            1
Tặng một số nội thất vd: Bộ bàn khách, bàn ông thần tài bếp.       1
Name: count, Length: 154, dtype: int64

In [934]:
def clean_phap_ly(df):
    df_result = df.copy()

    keywords = ['sdc', 'chung', 'hợp đồng mua bán', 'hđmb']
    pattern = '|'.join(keywords) # Tạo regex: "sdc|chung|..."

    condition = df_result['phap_ly'].str.contains(pattern, case=False, na=True)

    df_result['phap_ly'] = np.where(condition, 0, 1)

    return df_result

def clean_noi_that(df):
    df_result = df.copy()

    keywords = ['không', 'ko', 'thô']
    pattern = '|'.join(keywords)

    condition = df_result['noi_that'].str.contains(pattern, case=False, na=True)

    df_result['noi_that'] = np.where(condition, 0, 1)

    return df_result

In [935]:
df = clean_phap_ly(df)
df = clean_noi_that(df)
print('Đã xử lý và đồng bộ dữ liệu cột pháp lý và nội thất')

Đã xử lý và đồng bộ dữ liệu cột pháp lý và nội thất


## Xử lý cột quận/huyện

In [936]:
def clean_quan(df):
    mapping_quan = {
            'nguyễn thị thập': '7', 'Dương Cát Lợi': 'Nhà Bè',
            'Phường Nguyễn Cư Trinh': '1', 'Huyện Nhà Bè': 'Nhà Bè',
            'Quận Bình Tân': 'Bình Tân', 'Thành phố Thủ Đức': 'Thủ Đức',
            'Quận Tân Bình': 'Tân Bình'
        }
    df['quan_huyen'] = df['quan_huyen'].astype(str)
    df['quan_huyen'] = df['quan_huyen'].replace(mapping_quan)
    df['quan_huyen'] = df['quan_huyen'].str.replace('Quận ', '', regex=False)
    df['quan_huyen'] = df['quan_huyen'].str.strip()
    df = df[df['quan_huyen'] != 'Cần Giờ']
    return df
df = clean_quan(df)
print("Đã xử lý xong cột quận huyện")

Đã xử lý xong cột quận huyện


## Tạo cột 'khu_vuc' chứa Phường xã sau sát nhập (nếu khu vực nào ít th gom nhóm với các khu vực lân cận)

In [937]:
df['khu_vuc'] = df['phuong_xa']

### Quận 1

In [938]:
map_Q1 = {'Phường Bến Nghé':'Bến Thành/Sài Gòn', 'Phường Nguyễn Thái Bình': 'Bến Thành/Sài Gòn',
          'Phường Tân Định': 'Tân Định', 'Phường Đa Kao': 'Tân Định',
          'Phường Bến Thành': 'Bến Thành/Sài Gòn', 'Phường Phạm Ngũ Lão': 'Bến Thành/Sài Gòn',
          'Phường Cầu Ông Lãnh': 'Cầu Ông Lãnh', 'Phường Nguyễn Cư Trinh': 'Cầu Ông Lãnh',
          'Phường Cầu Kho': 'Cầu Ông Lãnh', 'Phường Cô Giang': 'Cầu Ông Lãnh', 'Đường Trần Đình Xu': 'Cầu Ông Lãnh'}
df.loc[df['quan_huyen'] == '1', 'khu_vuc'] = df.loc[df['quan_huyen'] == '1', 'phuong_xa'].replace(map_Q1)
df[df['quan_huyen']=='1']['khu_vuc'].value_counts()

khu_vuc
Cầu Ông Lãnh         135
Tân Định             123
Bến Thành/Sài Gòn     69
Name: count, dtype: int64

### Quận 3

In [939]:
map_Q3 = {'Phường 1': 'Bàn Cờ', 'Phường 2': 'Bàn Cờ', 'Phường 3': 'Bàn Cờ', 'Phường 5': 'Bàn Cờ',
          'Phường Võ Thị Sáu': 'Xuân Hòa', 'Phường 4': 'Xuân Hòa',
          'Phường 9': 'Nhiêu Lộc', 'Phường 10': 'Nhiêu Lộc', 'Phường 11': 'Nhiêu Lộc',
          'Phường 12': 'Nhiêu Lộc', 'Phường 13': 'Nhiêu Lộc', 'Phường 14': 'Nhiêu Lộc'}
df.loc[df['quan_huyen'] == '3', 'khu_vuc'] = df.loc[df['quan_huyen'] == '3', 'phuong_xa'].replace(map_Q3)
df[df['quan_huyen']=='3']['khu_vuc'].value_counts()

khu_vuc
Nhiêu Lộc    101
Xuân Hòa      98
Bàn Cờ        47
Name: count, dtype: int64

### Quận 4

In [940]:
df.loc[df['quan_huyen'] == '4', 'khu_vuc'] = 'Vĩnh Hội/Khánh Hội/Xóm Chiếu'
df[df['quan_huyen']=='4']['khu_vuc'].value_counts()

khu_vuc
Vĩnh Hội/Khánh Hội/Xóm Chiếu    87
Name: count, dtype: int64

### Quận 5

In [941]:
df.loc[df['quan_huyen'] == '5', 'khu_vuc'] = 'Chợ Quán/An Đông/Chợ Lớn'
df[df['quan_huyen']=='5']['khu_vuc'].value_counts()

khu_vuc
Chợ Quán/An Đông/Chợ Lớn    75
Name: count, dtype: int64

### Quận 6

In [942]:
df.loc[df['quan_huyen'] == '6', 'khu_vuc'] = 'Phú Lâm/Bình Phú/Bình Tiên/Bình Tây'
df[df['quan_huyen']=='6']['khu_vuc'].value_counts()

khu_vuc
Phú Lâm/Bình Phú/Bình Tiên/Bình Tây    33
Name: count, dtype: int64

### Quận 7

In [943]:
map_Q7 = {'Phường Bình Thuận': 'Tân Thuận', 'Phường Tân Thuận Đông': 'Tân Thuận', 'Phường Tân Thuận Tây': 'Tân Thuận',
          'Phường Phú Thuận': 'Phú Thuận', 'Phường Phú Mỹ': 'Tân Mỹ', 'Phường Tân Phú': 'Tân Mỹ',
          'Phường Tân Phong': 'Tân Hưng', 'Phường Tân Kiểng': 'Tân Hưng', 'Phường Tân Quy': 'Tân Hưng',
          'Phường Tân Hưng': 'Tân Hưng', 'Hồ Chí Minh': 'Tân Hưng', 'Tân Quy': 'Tân Hưng'}
df.loc[df['quan_huyen'] == '7', 'khu_vuc'] = df.loc[df['quan_huyen'] == '7', 'phuong_xa'].replace(map_Q7)
df[df['quan_huyen']=='7']['khu_vuc'].value_counts()

khu_vuc
Tân Hưng     113
Tân Thuận     97
Tân Mỹ        94
Phú Thuận     89
Name: count, dtype: int64

### Quận 8

In [944]:
map_Q8 = {'Phường 1': 'Chánh Hưng', 'Phường 2': 'Chánh Hưng', 'Phường 3': 'Chánh Hưng', 'Phường 4': 'Chánh Hưng',
          'Phường 5': 'Bình Chánh/Bình Hưng/Hưng Long/Bình Đông', 'Phường 6': 'Bình Chánh/Bình Hưng/Hưng Long/Bình Đông', 'Phường 7': 'Bình Chánh/Bình Hưng/Hưng Long/Bình Đông',
          'Phường 8': 'Chánh Hưng', 'Phường 9': 'Chánh Hưng', 'Phường 10': 'Chánh Hưng',
          'Phường 11': 'Phú Định', 'Phường 12': 'Phú Định', 'Phường 13': 'Phú Định', 'Phường 14': 'Phú Định', 'Phường 15': 'Phú Định', 'Phường 16': 'Phú Định'}
df.loc[df['quan_huyen'] == '8', 'khu_vuc'] = df.loc[df['quan_huyen'] == '8', 'phuong_xa'].replace(map_Q8)
df[df['quan_huyen']=='8']['khu_vuc'].value_counts()

khu_vuc
Chánh Hưng                                  64
Phú Định                                    45
Bình Chánh/Bình Hưng/Hưng Long/Bình Đông    28
Name: count, dtype: int64

### Quận 10

In [945]:
map_Q10 = {'Phường 1': 'Vườn Lài', 'Phường 2': 'Vườn Lài', 'Phường 3': 'Vườn Lài', 'Phường 4': 'Vườn Lài',
           'Phường 9': 'Vườn Lài', 'Phường 10': 'Vườn Lài', 'Phường 11': 'Vườn Lài',
          'Phường 12': 'Hoà Hưng', 'Phường 13': 'Hoà Hưng', 'Phường 15': 'Hoà Hưng',
          'Phường 5': 'Diên Hồng','Phường 6': 'Diên Hồng','Phường 7': 'Diên Hồng', 'Phường 8': 'Diên Hồng', 'Phường 14': 'Diên Hồng',}
df.loc[df['quan_huyen'] == '10', 'khu_vuc'] = df.loc[df['quan_huyen'] == '10', 'phuong_xa'].replace(map_Q10)
df[df['quan_huyen']=='10']['khu_vuc'].value_counts()

khu_vuc
Hoà Hưng     140
Vườn Lài     112
Diên Hồng     81
Name: count, dtype: int64

### Quận 11

In [946]:
df.loc[df['quan_huyen'] == '11', 'khu_vuc'] = 'Phú Thọ/Minh Phụng/Bình Thới/Hòa Bình'
df[df['quan_huyen']=='11']['khu_vuc'].value_counts()

khu_vuc
Phú Thọ/Minh Phụng/Bình Thới/Hòa Bình    47
Name: count, dtype: int64

### Quận 12

In [947]:
map_Q12 = {'Phường Tân Thới Nhất': 'Đông Hưng Thuận/Tân Thới Hiệp', 'Phường Đông Hưng Thuận': 'Đông Hưng Thuận/Tân Thới Hiệp', 'Phường Tân Hưng Thuận': 'Đông Hưng Thuận/Tân Thới Hiệp',
           'Phường Tân Chánh Hiệp': 'Trung Mỹ Tây', 'Phường Trung Mỹ Tây': 'Trung Mỹ Tây',
           'Phường Hiệp Thành': 'Đông Hưng Thuận/Tân Thới Hiệp', 'Phường Tân Thới Hiệp': 'Đông Hưng Thuận/Tân Thới Hiệp',
           'Phường Thạnh Xuân': 'Thới An', 'Phường Thới An': 'Thới An',
           'Phường Thạnh Lộc': 'An Phú Đông', 'Phường An Phú Đông': 'An Phú Đông'}
df.loc[df['quan_huyen'] == '12', 'khu_vuc'] = df.loc[df['quan_huyen'] == '12', 'phuong_xa'].replace(map_Q12)
df[df['quan_huyen']=='12']['khu_vuc'].value_counts()

khu_vuc
Thới An                          90
An Phú Đông                      86
Đông Hưng Thuận/Tân Thới Hiệp    61
Trung Mỹ Tây                     43
Name: count, dtype: int64

## Quận Bình Tân

In [948]:
map_BTN = {'Phường Bình Hưng Hòa B': 'Tân Tạo/Bình Tân',
           'Phường Tân Tạo': 'Tân Tạo/Bình Tân', 'Phường Tân Tạo A': 'Tân Tạo/Bình Tân',
           'Phường Bình Trị Đông B': 'An Lạc', 'Phường An Lạc A': 'An Lạc', 'Phường An Lạc': 'An Lạc',
           'Phường Bình Trị Đông': 'Bình Trị Đông', 'Phường Bình Trị Đông A': 'Bình Trị Đông',
           'Phường Bình Hưng Hòa': 'Bình Hưng Hòa', 'Phường Bình Hưng Hòa A': 'Bình Hưng Hòa',}
df.loc[df['quan_huyen'] == 'Bình Tân', 'khu_vuc'] = df.loc[df['quan_huyen'] == 'Bình Tân', 'phuong_xa'].replace(map_BTN)
df[df['quan_huyen']=='Bình Tân']['khu_vuc'].value_counts()

khu_vuc
Bình Hưng Hòa       100
Bình Trị Đông        82
An Lạc               81
Tân Tạo/Bình Tân     56
Name: count, dtype: int64

### Quận Bình Thạnh

In [949]:
map_BTH = {'Phường 1': 'Gia Định', 'Phường 2': 'Gia Định', 'Phường 3': 'Gia Định',
           'Phường 7': 'Gia Định','Phường 15': 'Gia Định', 'Phường 17': 'Gia Định',
           'Phường 6': 'Bình Thạnh', 'Phường 12': 'Bình Thạnh', 'Phường 14': 'Bình Thạnh', 'Phường 24': 'Bình Thạnh', 'Phường 26': 'Bình Thạnh',
           'Phường 5': 'Bình Lợi Trung', 'Phường 11': 'Bình Lợi Trung', 'Phường 13': 'Bình Lợi Trung',
           'Phường 19': 'Thạnh Mỹ Tây', 'Phường 21': 'Thạnh Mỹ Tây', 'Phường 22': 'Thạnh Mỹ Tây', 'Phường 25': 'Thạnh Mỹ Tây',
           'Phường 27': 'Bình Quới', 'Phường 28': 'Bình Quới',}
df.loc[df['quan_huyen'] == 'Bình Thạnh', 'khu_vuc'] = df.loc[df['quan_huyen'] == 'Bình Thạnh', 'phuong_xa'].replace(map_BTH)
df[df['quan_huyen']=='Bình Thạnh']['khu_vuc'].value_counts()

khu_vuc
Bình Lợi Trung    156
Gia Định          132
Bình Thạnh        131
Thạnh Mỹ Tây       98
Name: count, dtype: int64

### Quận Gò Vấp

In [950]:
map_GV = {'Phường 1': 'Hạnh Thông', 'Phường 3': 'Hạnh Thông', 'Phường 4': 'Hạnh Thông', 'Phường 7': 'Hạnh Thông',
           'Phường 5': 'An Nhơn','Phường 6': 'An Nhơn',
           'Phường 10': 'Gò Vấp', 'Phường 17': 'Gò Vấp',
           'Phường 13': 'An Hội Đông', 'Phường 15': 'An Hội Đông', 'Phường 16': 'An Hội Đông',
           'Phường 8': 'Thông Tây Hội', 'Phường 9': 'Thông Tây Hội', 'Phường 11': 'Thông Tây Hội',
           'Phường 12': 'An Hội Tây', 'Phường 14': 'An Hội Tây',}
df.loc[df['quan_huyen'] == 'Gò Vấp', 'khu_vuc'] = df.loc[df['quan_huyen'] == 'Gò Vấp', 'phuong_xa'].replace(map_GV)
df[df['quan_huyen']=='Gò Vấp']['khu_vuc'].value_counts()

khu_vuc
Thông Tây Hội    172
An Hội Tây       132
Hạnh Thông        60
An Hội Đông       54
An Nhơn           51
Gò Vấp            47
Name: count, dtype: int64

### Quận Phú Nhuận

In [951]:
map_PN = {'Phường 4': 'Đức Nhuận', 'Phường 5': 'Đức Nhuận', 'Phường 9': 'Đức Nhuận',
          'Phường 1': 'Cầu Kiệu', 'Phường 2': 'Cầu Kiệu', 'Phường 3': 'Cầu Kiệu', 'Phường 7': 'Cầu Kiệu', 'Phường 15': 'Cầu Kiệu',
          'Phường 8': 'Phú Nhuận', 'Phường 10': 'Phú Nhuận', 'Phường 11': 'Phú Nhuận', 'Phường 12': 'Phú Nhuận',
          'Phường 13': 'Phú Nhuận', 'Phường 14': 'Phú Nhuận', 'Phường 17': 'Phú Nhuận',}
df.loc[df['quan_huyen'] == 'Phú Nhuận', 'khu_vuc'] = df.loc[df['quan_huyen'] == 'Phú Nhuận', 'phuong_xa'].replace(map_PN)
df[df['quan_huyen']=='Phú Nhuận']['khu_vuc'].value_counts()

khu_vuc
Cầu Kiệu     193
Phú Nhuận    143
Đức Nhuận    110
Name: count, dtype: int64

### Quận Tân Bình

In [952]:
map_TB = {'Phường 1': 'Tân Sơn Hòa', 'Phường 2': 'Tân Sơn Hòa', 'Phường 3': 'Tân Sơn Hòa',
          'Phường 4': 'Tân Sơn Nhất', 'Phường 5': 'Tân Sơn Nhất', 'Phường 7': 'Tân Sơn Nhất',
          'Phường 6': 'Tân Hòa', 'Phường 8': 'Tân Hòa', 'Phường 9': 'Tân Hòa',
          'Phường 10': 'Bảy Hiền', 'Phường 11': 'Bảy Hiền', 'Phường 12': 'Bảy Hiền',
          'Phường 13': 'Tân Bình', 'Phường 14': 'Tân Bình', 'Phường 15': 'Tân Sơn',}
df.loc[df['quan_huyen'] == 'Tân Bình', 'khu_vuc'] = df.loc[df['quan_huyen'] == 'Tân Bình', 'phuong_xa'].replace(map_TB)
df[df['quan_huyen']=='Tân Bình']['khu_vuc'].value_counts()

khu_vuc
Tân Bình        133
Bảy Hiền        120
Tân Sơn          79
Tân Sơn Hòa      79
Tân Sơn Nhất     75
Tân Hòa          70
Name: count, dtype: int64

### Quận Tân Phú

In [953]:
map_TP = {'Phường Tây Thạnh': 'Tây Thạnh/Tân Sơn Nhì',
          'Phường Tân Sơn Nhì': 'Tây Thạnh/Tân Sơn Nhì', 'Phường Sơn Kỳ': 'Tây Thạnh/Tân Sơn Nhì', 'Tân Sơn Nhì': 'Tây Thạnh/Tân Sơn Nhì',
          'Phường Tân Quý': 'Phú Thọ Hòa', 'Phường Phú Thọ Hòa': 'Phú Thọ Hòa',
          'Phường Tân Thành': 'Tân Phú/Phú Thạnh', 'Phường Hòa Thạnh': 'Tân Phú/Phú Thạnh', 'Phường Phú Trung': 'Tân Phú/Phú Thạnh',
          'Phường Phú Thạnh': 'Tân Phú/Phú Thạnh', 'Phường Hiệp Tân': 'Tân Phú/Phú Thạnh', 'Phường Tân Thới Hòa': 'Tân Phú/Phú Thạnh'}
df.loc[df['quan_huyen'] == 'Tân Phú', 'khu_vuc'] = df.loc[df['quan_huyen'] == 'Tân Phú', 'phuong_xa'].replace(map_TP)
df[df['quan_huyen']=='Tân Phú']['khu_vuc'].value_counts()

khu_vuc
Tây Thạnh/Tân Sơn Nhì    93
Tân Phú/Phú Thạnh        85
Phú Thọ Hòa              84
Name: count, dtype: int64

### Quận 2, Quận 9, Thủ Đức

In [954]:
map_TD = {'Phường Hiệp Bình Chánh': 'Hiệp Bình', 'Phường Hiệp Bình Phước': 'Hiệp Bình',
          'Phường Bình Thọ': 'Thủ Đức', 'Phường Linh Chiểu': 'Thủ Đức', 'Phường Trường Thọ': 'Thủ Đức', 'Phường Linh Đông': 'Thủ Đức',
          'Phường Bình Chiểu': 'Tam Bình', 'Phường Tam Bình': 'Tam Bình', 'Phường Tam Phú': 'Tam Bình',
          'Phường Linh Xuân': 'Linh Xuân', 'Phường Linh Trung': 'Linh Xuân', 'Phường Linh Tây': 'Linh Xuân',
          'Phường Tăng Nhơn Phú A': 'Phước Long/Tăng Nhơn Phú', 'Phường Tăng Nhơn Phú B': 'Phước Long/Tăng Nhơn Phú', 'Phường Tân Phú': 'Phước Long/Tăng Nhơn Phú', 'Phường Hiệp Phú': 'Phước Long/Tăng Nhơn Phú',
          'Phường Long Bình': 'Long Phước/Long Bình', 'Phường Long Thạnh Mỹ': 'Long Phước/Long Bình',
          'Phường Trường Thạnh': 'Long Phước/Long Bình', 'Phường Long Phước': 'Long Phước/Long Bình',
          'Phường Long Trường': 'Long Trường', 'Phường Phú Hữu': 'Long Trường',
          'Phường Thạnh Mỹ Lợi': 'Cát Lái', 'Phường Cát Lái': 'Cát Lái',
          'Phường Bình Trưng Đông': 'Bình Trưng', 'Phường Bình Trưng Tây': 'Bình Trưng', 'Phường An Phú': 'Bình Trưng',
          'Phường Phước Long A': 'Phước Long/Tăng Nhơn Phú', 'Phường Phước Long B': 'Phước Long/Tăng Nhơn Phú', 'Phường Phước Bình': 'Phước Long/Tăng Nhơn Phú',
          'Phường An Khánh': 'An Khánh', 'Phường Thủ Thiêm': 'An Khánh', 'Phường An Lợi Đông': 'An Khánh', 'Phường Thảo Điền': 'An Khánh','Phường Bình An': 'An Khánh',}
mask_TD = df['quan_huyen'].isin(['2', '9', 'Thủ Đức'])
df.loc[mask_TD, 'khu_vuc'] = df.loc[mask_TD, 'phuong_xa'].replace(map_TD)
df[df['quan_huyen'].isin(['2', '9', 'Thủ Đức'])]['khu_vuc'].value_counts()

khu_vuc
Hiệp Bình                   138
Thủ Đức                     122
Phước Long/Tăng Nhơn Phú    113
Long Trường                  99
Bình Trưng                   97
An Khánh                     75
Long Phước/Long Bình         72
Cát Lái                      49
Linh Xuân                    49
Tam Bình                     40
Name: count, dtype: int64

### Bình Chánh

In [955]:
map_BC = {'Xã Vĩnh Lộc A': 'Vĩnh Lộc/Tân Nhựt/Bình Lợi', 'Xã Vĩnh Lộc B': 'Vĩnh Lộc/Tân Nhựt/Bình Lợi', 'Xã Phạm Văn Hai': 'Vĩnh Lộc/Tân Nhựt/Bình Lợi',
          'Xã Lê Minh Xuân': 'Vĩnh Lộc/Tân Nhựt/Bình Lợi', 'Xã Bình Lợi': 'Vĩnh Lộc/Tân Nhựt/Bình Lợi',
          'Xã Tân Túc': 'Vĩnh Lộc/Tân Nhựt/Bình Lợi', 'Xã Tân Nhựt': 'Vĩnh Lộc/Tân Nhựt/Bình Lợi', 'Xã Tân Kiên': 'Vĩnh Lộc/Tân Nhựt/Bình Lợi', 'Thị trấn Tân Túc': 'Vĩnh Lộc/Tân Nhựt/Bình Lợi',
          'Xã Tân Quý Tây': 'Bình Chánh/Bình Hưng/Hưng Long/Bình Đông', 'Xã Bình Chánh': 'Bình Chánh/Bình Hưng/Hưng Long/Bình Đông', 'Xã An Phú Tây': 'Bình Chánh/Bình Hưng/Hưng Long/Bình Đông',
          'Xã Đa Phước': 'Bình Chánh/Bình Hưng/Hưng Long/Bình Đông', 'Xã Hưng Long': 'Bình Chánh/Bình Hưng/Hưng Long/Bình Đông', 'Xã Qui Đức': 'Bình Chánh/Bình Hưng/Hưng Long/Bình Đông',
          'Xã Phong Phú': 'Bình Chánh/Bình Hưng/Hưng Long/Bình Đông', 'Xã Bình Hưng': 'Bình Chánh/Bình Hưng/Hưng Long/Bình Đông',}
df.loc[df['quan_huyen'] == 'Bình Chánh', 'khu_vuc'] = df.loc[df['quan_huyen'] == 'Bình Chánh', 'phuong_xa'].replace(map_BC)
df[df['quan_huyen']=='Bình Chánh']['khu_vuc'].value_counts()

khu_vuc
Vĩnh Lộc/Tân Nhựt/Bình Lợi                  34
Bình Chánh/Bình Hưng/Hưng Long/Bình Đông    25
Name: count, dtype: int64

### Nhà Bè

In [956]:
map_NB = {'Thị trấn Nhà Bè': 'Nhà Bè', 'Xã Phú Xuân': 'Nhà Bè', 'Xã Phước Kiển': 'Nhà Bè', 'Xã Phước Lộc': 'Nhà Bè',
          'Xã Nhơn Đức': 'Hiệp Phước', 'Xã Hiệp Phước': 'Hiệp Phước', 'Xã Long Thới': 'Hiệp Phước', }
df.loc[df['quan_huyen'] == 'Nhà Bè', 'khu_vuc'] = df.loc[df['quan_huyen'] == 'Nhà Bè', 'phuong_xa'].replace(map_NB)
df[df['quan_huyen']=='Nhà Bè']['khu_vuc'].value_counts()

khu_vuc
Nhà Bè        322
Hiệp Phước    131
Name: count, dtype: int64

### Hóc Môn - Củ Chi

In [957]:
df.loc[df['quan_huyen'] == 'Hóc Môn', 'khu_vuc'] = 'Hóc Môn/Củ Chi'
df.loc[df['quan_huyen'] == 'Củ Chi', 'khu_vuc'] = 'Hóc Môn/Củ Chi'
df[df['phuong_xa']=='Hóc Môn/Củ Chi']['khu_vuc'].value_counts()

Series([], Name: count, dtype: int64)

In [958]:
df['khu_vuc'].value_counts()

khu_vuc
Nhà Bè                                 322
Cầu Kiệu                               193
Thông Tây Hội                          172
Bình Lợi Trung                         156
Phú Nhuận                              143
                                      ... 
Phú Định                                45
Trung Mỹ Tây                            43
Tam Bình                                40
Vĩnh Lộc/Tân Nhựt/Bình Lợi              34
Phú Lâm/Bình Phú/Bình Tiên/Bình Tây     33
Name: count, Length: 64, dtype: int64

## Xử lý dữ liệu khuyết

In [959]:
df_clean = df.dropna(subset=['dien_tich', 'so_pn', 'so_tang', 'mat_tien','duong_vao','gia'])
print('Đã xóa các dòng bị thiếu dữ liệu')
print('Số bản ghi còn lại:', len(df_clean))
df_clean.loc[df['so_wc'].isnull(), 'so_wc'] = df_clean['so_pn']

Đã xóa các dòng bị thiếu dữ liệu
Số bản ghi còn lại: 3450


## Xử lý ngoại lai

In [960]:
df_clean['gia/dien_tich'] = df_clean['gia']/ df_clean['dien_tich']
def remove_outliers_by_unit_price(df, unit_price_col='gia/dien_tich', group_col='khu_vuc'):
    """
    Hàm lọc ngoại lai dựa trên cột đơn giá đã có sẵn.
    """
    # Tính toán ngưỡng IQR cho đơn giá theo từng phường
    # transform giúp tính toán mà không làm thay đổi cấu trúc DataFrame
    q1 = df.groupby(group_col)[unit_price_col].transform(lambda x: x.quantile(0.25))
    q3 = df.groupby(group_col)[unit_price_col].transform(lambda x: x.quantile(0.75))
    iqr = q3 - q1

    # Xác định biên độ (thường dùng 1.5, hoặc 3.0 nếu muốn giữ nhiều dữ liệu hơn)
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    # Lọc dữ liệu
    df_clean = df[
        (df[unit_price_col] >= lower_bound) &
        (df[unit_price_col] <= upper_bound)
    ].copy()

    return df_clean

df_final = remove_outliers_by_unit_price(df_clean, unit_price_col='gia/dien_tich')
df_final = df_final[(df_final['duong_vao'] >= 2) &  (df_final['duong_vao'] <= 30)]
df_final = df_final[(df_final['mat_tien'] > 2.5) & (df_final['mat_tien'] <= 25)]
df_final = df_final[df_final['dien_tich'] <= 400]
df_final = df_final[df_final['gia'] < 1e11]
df_final = df_final[df_final['so_tang'] < 7]
# Kiểm tra kết quả
print(f"Số lượng trước khi lọc: {len(df_clean)}")
print(f"Số lượng sau khi lọc: {len(df_final)}")

Số lượng trước khi lọc: 3450
Số lượng sau khi lọc: 3252


## Dữ liệu sau xử lý

In [961]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 3252 entries, 0 to 6005
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   phuong_xa      3252 non-null   str    
 1   quan_huyen     3252 non-null   str    
 2   gia            3252 non-null   float64
 3   dien_tich      3252 non-null   float64
 4   so_pn          3252 non-null   float64
 5   so_wc          3252 non-null   float64
 6   so_tang        3252 non-null   float64
 7   mat_tien       3252 non-null   float64
 8   duong_vao      3252 non-null   float64
 9   phap_ly        3252 non-null   int64  
 10  noi_that       3252 non-null   int64  
 11  khu_vuc        3252 non-null   str    
 12  gia/dien_tich  3252 non-null   float64
dtypes: float64(8), int64(2), str(3)
memory usage: 355.7 KB


In [962]:
df_final

,phuong_xa,quan_huyen,gia,dien_tich,so_pn,so_wc,so_tang,mat_tien,duong_vao,phap_ly,noi_that,khu_vuc,gia/dien_tich
0,Thị trấn Tân Túc,Bình Chánh,1.470000e+10,173.3,3.0,2.0,1.0,4.4,16.0,1,0,Vĩnh Lộc/Tân Nhựt/Bình Lợi,8.482400e+07
2,Thị trấn Tân Túc,Bình Chánh,6.000000e+09,101.0,2.0,3.0,2.0,5.0,6.0,1,0,Vĩnh Lộc/Tân Nhựt/Bình Lợi,5.940594e+07
4,Xã Bình Hưng,Bình Chánh,1.890000e+10,193.0,2.0,2.0,2.0,10.0,12.0,1,0,Bình Chánh/Bình Hưng/Hưng Long/Bình Đông,9.792746e+07
5,Xã Bình Hưng,Bình Chánh,2.100000e+10,100.0,6.0,7.0,5.0,5.0,10.0,1,1,Bình Chánh/Bình Hưng/Hưng Long/Bình Đông,2.100000e+08
6,Xã Bình Hưng,Bình Chánh,4.100000e+09,45.4,2.0,2.0,2.0,3.9,2.0,1,0,Bình Chánh/Bình Hưng/Hưng Long/Bình Đông,9.030837e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6000,Phường Trường Thọ,Thủ Đức,1.650000e+10,122.0,6.0,6.0,6.0,5.0,10.0,1,1,Thủ Đức,1.352459e+08
6001,Phường Trường Thọ,Thủ Đức,1.200000e+10,105.3,2.0,2.0,2.0,5.0,10.0,1,0,Thủ Đức,1.139601e+08
6003,Phường Trường Thọ,Thủ Đức,7.700000e+09,50.6,2.0,2.0,2.0,4.4,6.0,1,1,Thủ Đức,1.521739e+08
6004,Phường Trường Thọ,Thủ Đức,7.600000e+09,61.0,4.0,2.0,3.0,4.0,4.0,1,0,Thủ Đức,1.245902e+08


## Xuất data đã xử lý

In [963]:
df_final['phuong_xa'] = df_final['khu_vuc']

In [964]:
df_final = df_final.drop(columns=['quan_huyen', 'gia/dien_tich', 'khu_vuc'])
df_final.to_csv('data_clean.csv', index=False)